# NS-Spec v0.2 — Neuro-Symbolic Specification-First Development

**Status:** Reworked design — supersedes v0.1.0  
**Primary solver:** Z3, reached *exclusively* through the existing SPUR `solve` MCP  
**Authoring surface:** Markdown + `spec-yaml` fences  
**Future render layer:** MAS

> v0.2 **removes the Python symbolic implementation adapter** from v0.1 and
> reconceives NS-Spec as a **specification front-end + audit rule-set +
> conformance-test oracle, layered over the existing `solve` MCP**. All Z3 work
> flows through `solve_constraints` / `solve_smt` / `get_solve_result`; NS-Spec
> never binds Z3 directly.

### Contents
1. Executive summary — what changed and why
2. Architecture
3. Revised core principles
4. Verification decision flow + the determinism flaw
5. Verification menu (solver probes + structural gates)
6. Conformance testing replaces symbolic refinement
7. `solve` MCP integration + brain→worker handoff
8. Spec lifecycle and statuses
9. Section delete / keep / add map, logic contract, milestones
10. Appendix — reproduced `solve` evidence (10 runs)

## 1. Executive summary

NS-Spec governs AI-generated business logic with an **executable specification**.
A human (or agent) writes a Markdown spec; NS-Spec parses it into a typed
Canonical IR; Z3 audits the spec *before* implementation; a human approves the
normalized semantics; then the implementation is checked — **not by symbolic
extraction, but by Z3-generated conformance tests** run black-box against the
real code.

### What changed from v0.1

| Aspect | v0.1 | v0.2 |
|---|---|---|
| Implementation check | Symbolic Python AST → Z3 refinement | **Z3-generated conformance tests**, any language |
| Z3 binding | Bespoke (parallel to SPUR) | **Reuse `solve` MCP** (`solve_constraints` / `solve_smt`) |
| Determinism check | Opt-in (`deterministic: true`) | **Default-on** |
| Postcondition idiom | Output-guarded (`when status==success`) | **Input-guarded determination** (default) |
| `Real` type | MUST (unsound vs Python float) | SHOULD (conformance oracle + tolerance) |
| Logic contract | "QF_LIA" (silently escapes via `/`, `%`, `x*y`) | **QF_LIA + Bool + enum only**; nonlinear → `unsupported-term` |
| Acceptance claim | `verified` | `conformance-passed` (bounded) vs `refinement-proven` (optional, MVP2+) |

### Why the symbolic adapter was removed

The v0.1 Python symbolic executor was the most expensive, highest-risk,
Python-only component — and its "verified" claim was only as trustworthy as a
brand-new path extractor (a false `unsat` there is a **false proof**, worse than
`unknown`). Its job is done better and language-agnostically by conformance
testing on Z3-generated boundary models, and its Z3 work is already shipped in
SPUR's `solve` MCP. Building either again is pure duplication.

## 2. Architecture

```mermaid
flowchart TD
    A[spec.md files Markdown plus spec-yaml fences] --> B[Markdown Front End fence extractor safe-YAML source-locs]
    B --> C[(Canonical Spec IR types ops props source-maps)]
    C --> D[Spec Audit Layer lowers IR to B-prime and SMT]
    C --> E[Conformance Test Generator Z3 witness and boundary models]
    C --> F[Proof Report Builder]
    D -. calls .-> S((solve MCP solve_constraints and solve_smt))
    E -. calls .-> S
    S -. persist solve_id .-> H[get_solve_result CI and brain-to-worker handoff]
    E --> T[generated tests pytest any language]
    T --> I[Real implementation any language incl IO]
    D --> F
    E --> F
    F --> P[beads plus CI gate]
```

**No "Implementation Adapter (Python AST)" box exists in v0.2.** The Canonical
IR feeds three consumers — audit, conformance generation, and reporting — all of
which call the shared `solve` MCP. The implementation sits *downstream* of
generated tests; it never enters the solver.

## 3. Revised core principles

1. **Markdown is the authoring surface.** `spec-yaml` fences carry formal
   semantics; surrounding prose is informative.
2. **The Canonical Spec IR is the single semantic source of truth.** Z3 input,
   conformance tests, proof reports, and future MAS views all derive from it.
3. **Solver-neutral at the spec layer; Z3 at the engine layer.** The spec
   language exposes no Z3 API. The IR lowers to B′ (`solve_constraints`) or SMT
   (`solve_smt`) — both already shipped in SPUR.
4. **The verifier is the acceptance authority.** Agents draft, implement,
   repair, and generate tests; they never decide whether verification passed.
5. **Determinism is a default gate, not opt-in.** Every operation is assumed
   deterministic unless it declares `nondeterministic: true`. *(v0.1 made this
   opt-in and its own flagship example silently failed it — see §4.)*
6. **Postconditions determine outputs from inputs.** The input-guarded idiom is
   the default: `when <input condition>: status == <value>`. Output-guarded
   posts (`when status == success`) *describe*; they do not *determine*.
7. **Unknown is not success.** `unknown`, `timeout`, `unsupported`, parse/type
   failure, and internal error all fail the gate.
8. **Bounded is not proven.** Conformance testing yields `conformance-passed`,
   never `verified`. `refinement-proven` is reserved for the optional direct-
   to-SMT lowering path (MVP2+).

## 4. Verification decision flow

The semantic probes run as `solve` calls, with structural gates around them.
Order matters: cheap structural checks first, then the high-signal semantic
checks.

```mermaid
flowchart TD
    S([spec-yaml block]) --> P{parse and schema}
    P -- error --> X2([exit 2 parse])
    P -- ok --> T{type check}
    T -- error --> X3([exit 3 type])
    T -- ok --> NV{non-vacuity SAT assumes and requires}
    NV -- unsat --> XV([vacuous spec])
    NV -- sat --> C{consistency SAT all posts and invariants}
    C -- unsat --> XC([conflicting subset via subset-iteration])
    C -- sat --> D{determinism counterexample SAT}
    D -- sat --> XD([under-determined fix or opt out])
    D -- unsat --> G{input-guard gap SAT}
    G -- sat --> XG([uncovered input region])
    G -- unsat --> O{input-guard overlap SAT}
    O -- sat --> XO([ambiguous status partition])
    O -- unsat --> CT{constructive totality complete}
    CT -- no --> XT([non-total output relation])
    CT -- yes --> BR{all declared status branches reachable}
    BR -- no --> XBR([dead branch diagnostic])
    BR -- yes --> OK([spec approved])
```

**Reordered emphasis vs v0.1.** v0.1 led with *consistency* (§16.2), which is
almost always trivially `sat` and catches little. The checks that actually find
bugs are **non-vacuity, determinism, and output-determination** — so they get
first-class billing here.

## 4b. The determinism flaw — and the fix

The spec's own flagship example (`TRANSFER-001`) **fails its own §16.5
determinism definition**, because the happy path is **output-guarded**:

```yaml
# v0.1 (NON-deterministic) — describes success, never requires it
- when: status == success
  expr: source_after == source_balance - amount
```

This says "*if* you succeeded, the math holds." It never says *when success is
permitted*, so in the eligible region every status is consistent. Z3 confirms it
(see Appendix): the determinism check `Pre ∧ Post(x,y1) ∧ Post(x,y2) ∧ y1≠y2`
returns **`sat`** — two distinct valid outputs for one input.

```mermaid
flowchart LR
    subgraph ORIG[ORIGINAL output-guarded Z3 sat non-deterministic]
        I1[input amount 1 src 3 tgt 0] --> A1[status success sa 2 ta 1]
        I1 --> A2[status insufficient sa 3 ta 0]
        I1 --> A3[status invalid sa 3 ta 0]
    end
    subgraph FIX[CORRECTED input-guarded Z3 unsat deterministic]
        I2[same input] --> B1[status success ONLY sa 2 ta 1]
    end
```

### The fix — input-guarded, exhaustive determination

```yaml
ensures:
  - id: POST-INVALID-001
    when: amount <= 0
    expr: status == invalid_amount
  - id: POST-INSUFFICIENT-001
    when: amount > 0 and source_balance < amount
    expr: status == insufficient_balance
  - id: POST-SUCCESS-001          # determines status FROM inputs
    when: amount > 0 and source_balance >= amount
    expr: status == success
  - id: POST-SUCCESS-002          # output relations still guarded on status
    when: status == success
    expr: source_after == source_balance - amount
  - id: POST-SUCCESS-003
    when: status == success
    expr: target_after == target_balance + amount
  - id: POST-FAILURE-UNCHANGED-001
    when: status != success
    expr: source_after == source_balance
  - id: POST-FAILURE-UNCHANGED-002
    when: status != success
    expr: target_after == target_balance
```

Under this contract the determinism counterexample formula returns **`unsat`** —
a proof of determinism for the encoded relation. Separate gap and overlap probes
also return `unsat`, proving the three status guards form an exhaustive,
mutually exclusive input partition. That establishes **status determination**;
full relation totality still requires every branch to define every output. A
`sat` assignment is a counterexample witness, not a golden model; only the
encoded predicate and status are stable.

## 5. Verification menu — solver probes + structural gates

| Check (§16) | Encoding / gate | Pass | Fail |
|---|---|---|---|
| Non-vacuity | `SAT(assumptions ∧ requires)` → `solve_constraints` | `sat` | `unsat` (vacuous) |
| Consistency | `SAT(requires ∧ ensures ∧ invariants)` → `solve_constraints` | `sat` | `unsat` → client-side subset-iteration to isolate conflicting property IDs (unsat-core is gate-rejected; see Appendix B) |
| **Determinism** (default-on) | `SAT(Pre ∧ Post(x,y1) ∧ Post(x,y2) ∧ y1≠y2)` | `unsat` | `sat` = under-determined |
| Status-partition coverage | `SAT(pre ∧ ¬(g1 ∨ … ∨ gn))` | `unsat` | `sat` = uncovered input region |
| Status-partition exclusivity | `SAT(pre ∧ ⋁i<j(gi ∧ gj))` | `unsat` | `sat` = overlapping status guards |
| **Constructive totality** | structural: every exhaustive input branch defines every output using total, well-typed QF terms | complete | missing output or partial term |
| Branch reachability | per-enum-value `SAT(pre ∧ posts ∧ status=v)` | `sat` = reachable | `unsat` = dead branch; **not a totality failure** |
| Invariant initiation (SM) | `SAT(initial ∧ ¬invariant)` | `unsat` | `sat` |
| Invariant preservation (SM) | `SAT(inv(state) ∧ guard ∧ update ∧ ¬inv(next))` | `unsat` | `sat` = transition breaks invariant |
| Bounded reachability | k-step unroll, **labeled bounded** | `unsat`/`sat` | never presented as unbounded |

Proof obligations use the assert-negation pattern: `unsat` proves that the
encoded counterexample cannot exist; `sat` returns a concrete counterexample.
Per-enum-value satisfiability is a branch-reachability/dead-code audit, not a
proof of `∀x∃y.Post(x,y)`. Finite expansion eliminates the enum choice but not
existential integer outputs. MVP therefore gates on constructive totality;
quantified relational totality may be attempted through `solve_smt`, but
`unknown` or `timeout` still fails the proof gate.

**Logic contract (v0.2):** the QF_LIA fragment + Bool + enum only — the `solve`
engine actually emits `(set-logic QF_NIA)` (it supports nonlinear `mul`), so
NS-Spec restricts specs to the linear fragment to keep every audit check a real
`sat`/`unsat` rather than `unknown`. Nonlinear terms (`x*y` with two variables),
floor-division and modulo leave the linear fragment and are reported as
`unsupported-term` — **never silently degraded to `unknown`**.

## 6. Conformance testing replaces symbolic refinement

With the symbolic adapter deleted, "does the implementation satisfy the spec?"
is answered by **executing the real implementation against `solve`-generated
tests**. The spec is the oracle; Z3 is reached only through the existing solver
service.

```mermaid
flowchart TD
    SPEC[Approved Canonical IR] --> Q1[Z3 one witness per enum output]
    SPEC --> Q2[Z3 boundary models 0 1 and edges]
    SPEC --> Q3[Z3 invariant-edge witnesses]
    SPEC --> Q4[Z3 seeded-mutation witnesses]
    Q1 --> GEN[test emitter input plus expected output]
    Q2 --> GEN
    Q3 --> GEN
    Q4 --> GEN
    GEN --> FILES[generated tests spec-property py]
    FILES --> RUN[pytest vs real implementation]
    RUN -- mismatch --> CEX[failure to repair]
    RUN -- all pass --> COV[conformance-passed bounded not verified]
```

The generator requests boundaries with explicit constraints (`x=0`, `x=1`,
`x=edge±1`, and property boundaries). A plain `sat` model is arbitrary and MUST
NOT be described as minimized or boundary-selected unless those predicates were
part of the solve request.

### Why this beats symbolic refinement

| Concern | Symbolic adapter (v0.1, deleted) | Conformance via `solve` MCP (v0.2) |
|---|---|---|
| Language | Python only | any |
| `Real`/float soundness | **unsound** (Z3 Real ≠ float) | exercised for real (oracle + tolerance) |
| False-proof / false-acceptance risk | **high** (buggy path extraction → false `unsat`) | no proof claim; false acceptance remains possible if IR lowering or the generated oracle is wrong |
| I/O, loops, side effects | rejected (`unsupported`) | testable through a wrapper |
| New code to build and validate | large symbolic executor | Markdown front end + Canonical IR + lowering + generator; solver runtime reused |
| Guarantee delivered | `verified` (overclaimed) | `conformance-passed on N Z3 models` (honest) |

`conformance-passed` is a **distinct status** from `refinement-proven`. The
latter is reserved for an optional, per-language direct-to-`solve_smt` AST
lowering whose supported subset must itself be proved sound — deferred to
MVP2+.

## 7. `solve` MCP integration + brain→worker handoff

NS-Spec owns the **front-end** (Markdown, IR, diagnostics, approval, CI gate);
the existing `solve` MCP owns the **solver**. NS-Spec lowers the IR to B′ or
SMT and calls `solve_constraints` / `solve_smt`, reusing `persist: solve_id` +
`get_solve_result` for CI and brain→worker handoff. **No new Z3 binding.**

```mermaid
sequenceDiagram
    participant B as Brain
    participant S as solve MCP Z3
    participant W as Worker
    participant CI as CI gate
    B->>S: solve_constraints IR-to-B-prime persist true
    S-->>B: solve_id plus sat or unsat
    B->>W: delegate task plus solve_id audit
    W->>S: get_solve_result solve_id
    S-->>W: authoritative model
    W->>W: lower spec to conformance tests
    W->>CI: PR plus generated tests
    CI->>S: re-run audit and conformance
    S-->>CI: proof report
    CI-->>CI: merge or block
```

The handoff reuses SPUR's existing solver-persistence contract: the brain
solves once with `persist: true`, embeds the `solve_id` in the worker task, and
the worker reloads it as **authoritative** via `get_solve_result` — never
re-inventing constants or re-solving.

## 8. Spec lifecycle and statuses

```mermaid
stateDiagram-v2
    [*] --> draft
    draft --> parsed: fence and YAML ok
    parsed --> structurally_valid: schema ok
    structurally_valid --> semantically_valid: type check and audit
    semantically_valid --> approved: human sign-off
    approved --> implementation_pending
    implementation_pending --> conformance_passed: tests green
    implementation_pending --> refinement_proven: optional SMT lowering MVP2 onward
    conformance_passed --> [*]
    refinement_proven --> [*]
    semantically_valid --> verification_failed: audit unsat or under-determined
    implementation_pending --> verification_failed: conformance mismatch
    verification_failed --> draft: repair
```

**Key change vs v0.1:** the terminal acceptance state splits into
`conformance_passed` (default, bounded) and `refinement_proven` (optional,
proof-grade). v0.1's single `verified` state overclaimed; v0.2 never labels a
bounded result as a proof. Modifying an approved formal block changes its
canonical IR hash and forces a return to `draft`.

## 9. Section map, logic contract, milestones

### 9.1 Delete / keep / add vs v0.1

| v0.1 section | v0.2 action |
|---|---|
| §17 Implementation verification (symbolic AST) | **DELETE** — replaced by §6 conformance testing |
| §17.1 restricted-Python subset | **DELETE** — conformance tests target any callable |
| §8 / §17.2 `mode: symbolic-ast` | **REPLACE** with `mode: conformance-test` (default) |
| Milestone 4 (Python symbolic adapter) | **REPLACE** with conformance-test generator + runner |
| §11.1 `Real` (MUST) | **DEMOTE** to SHOULD (oracle tolerance) |
| §16.5 determinism (opt-in) | **ELEVATE** to default-on gate |
| §16.2 consistency (leads the section) | **DE-EMPHASIZE** — lead with non-vacuity / determinism |
| §16.4 totality (`∀x∃y`) | **REPLACE** in MVP with constructive totality; keep enum-value SAT as a separate branch-reachability audit |
| §12.1 `/`, `%`, nonlinear | **FORBID** in v0.1 scope → `unsupported-term` |
| §5–16 spec language + IR + audit | **KEEP** — the genuine heart |
| §19 regression-test generation | **ELEVATE** to primary implementation check |
| — (new) | **ADD** solver-backend section: lower IR → `solve` MCP |
| — (new) | **ADD** `conformance_passed` vs `refinement_proven` statuses |

### 9.2 Layering

```mermaid
flowchart TD
    subgraph NS[NS-Spec application layer]
        FE[Markdown FE and spec-yaml parser]
        IRM[Canonical IR and type checker]
        AUD[Spec audit rules]
        CFG[Conformance test generator]
        RPT[Proof report]
    end
    subgraph EXIST[SPUR existing]
        SOLVE[solve MCP solve_constraints solve_smt get_solve_result]
        PMPM[beads PM and CI]
    end
    FE --> IRM --> AUD --> SOLVE
    IRM --> CFG --> SOLVE
    AUD --> RPT
    CFG --> RPT
    RPT --> PMPM
```

### 9.3 Revised milestones

1. **Markdown front end** — fence extraction, safe-YAML, schema, source-locs.
2. **Canonical Spec IR** — types, enums, ops, expression parser, type checker.
3. **Spec audit over `solve`** — lower IR → B′/SMT; non-vacuity, consistency,
   determinism (default-on), partition gap/overlap, constructive totality,
   branch reachability, and invariants; conflicting-subset via client-side
   subset-iteration (unsat-core gate-rejected).
4. **Conformance-test generator** — explicit boundary predicates plus `solve`
   witness and seeded-mutation models → generated tests.
5. **Agent + CI integration** — MCP server, CLI, AGENTS.md policy, GitHub Actions,
   end-to-end repair loop. *(No symbolic-adapter milestone.)*

## 10. Appendix — reproduced `solve` evidence (10 runs)

All runs were executed on 2026-07-29 through the existing `solve_constraints`
MCP tool using B′ only. Runs #2–8 included the transfer input requirements,
postcondition implications, non-negative output invariants, and balance
conservation; run #1 intentionally encoded a mutant that violates them. `sat`
models below are concrete witnesses, not golden assignments;
Z3 may return different witnesses for the same satisfiable predicate. Transient
wall-clock timings are intentionally omitted.

| # | Check | Encoding (B′) | Reproduced result |
|---|---|---|---|
| 1 | Seeded mutation witness | insufficient input plus a mutant that returns `success` and applies the transfer | **sat** → `src=6,tgt=0,amt=8,status=success,sa=-2,ta=8` |
| 2 | **Determinism, ORIGINAL contract** | `pre ∧ Post(x,y1) ∧ Post(x,y2) ∧ y1≠y2` | **sat** → at `src=2,tgt=1,amt=1`, `insufficient`/unchanged and `success`/transferred are both valid |
| 3 | **Determinism, CORRECTED contract** | same counterexample formula with input-guarded posts | **unsat** → no two distinct outputs exist |
| 4 | Status-partition gap | `pre ∧ ¬(g_invalid ∨ g_insufficient ∨ g_success)` | **unsat** → guards are exhaustive |
| 5 | Status-partition overlap | `pre ∧ ⋁i<j(gi ∧ gj)` | **unsat** → guards are mutually exclusive |
| 6 | Conformance witness — `success` | `pre ∧ corrected_contract ∧ status=success` | **sat** → `src=7,tgt=7,amt=5 ⇒ sa=2,ta=12` |
| 7 | Conformance witness — `invalid_amount` | `pre ∧ corrected_contract ∧ status=invalid_amount` | **sat** → `src=3,tgt=0,amt=-8 ⇒ unchanged` |
| 8 | Conformance witness — `insufficient_balance` | `pre ∧ corrected_contract ∧ status=insufficient_balance` | **sat** → `src=3,tgt=0,amt=8 ⇒ unchanged` |
| 9 | SHIP preservation, original guard | `inv ∧ status=paid ∧ ship_update ∧ ¬inv(next)` | **sat** → `captured=0,next_status=shipped` |
| 10 | SHIP preservation, strengthened guard | run #9 plus `captured>0` | **unsat** → strengthened transition is inductive |

### What this proves

- Runs #2–3 reproduce the determinism defect and prove the corrected encoded
  relation deterministic via the assert-negation pattern.
- Runs #4–5 separately prove status-partition coverage and exclusivity; neither
  result is a substitute for full relation totality.
- Runs #6–8 establish that every declared transfer status has at least one
  witness suitable for a generated test. They remain bounded examples, not a
  refinement proof.
- Run #1 is an explicit seeded mutation, not symbolic inspection of a real
  implementation. It demonstrates a test-generation target without reviving the
  deleted implementation adapter.
- Runs #9–10 reproduce the distinction between invariant preservation and
  reachability discussed in Appendix C2.

### Bottom line

The solver results validate the logical correction and the existing `solve` MCP
integration. They do **not** validate the not-yet-implemented Markdown→IR
lowering or generated-test runner; those components require their own
property-preserving and end-to-end conformance tests.

---

## Appendix B — Grounding against `crates/spur-solver` (verified)

The solver-integration claims were checked against the real solver source via code-explore, and the unsat-core gap was confirmed by execution:

| Claim | Source | Status |
|---|---|---|
| B′ `Variable` / `ConstraintExpr` / `ConstraintOp` grammar (vars, tagged expr, ops, no `div`) | `spur-solver/src/types.rs:30-179` | exact match |
| Status envelope `Sat` / `Unsat` / `Unknown` / `Timeout` (+`Error`) | `types.rs:320-334` | match |
| Encoder hardcodes `(set-logic QF_NIA)` — nonlinear `mul` supported | `encode.rs:125` | reframes QF_LIA as a self-imposed fragment |
| SMT gate allowlist = `set-logic` / `assert` / `check-sat` / `get-model` / `get-value` / `push` / `pop` / `declare-*` | `smt_gate.rs:143-153` | **`get-unsat-core` REJECTED** → client-side subset-iteration |
| `unknown` never collapsed to `unsat` | test `fake_solver_unknown_is_not_collapsed_into_unsat` | enforced in code |
| `persist` → `solve_id`, worker-side solve tools | `spur-core/src/worker_server.rs`, test `solve_constraints_persists_when_requested` | wired + tested |

The `get-unsat-core` rejection was reproduced live: a `solve_smt` script containing it returns `command get-unsat-core at byte 71 is not allowed`. Net: NS-Spec ships as a **pure client of the existing `solve` MCP with zero changes to `spur-solver`**; widening the gate for native unsat-cores is a future optimization, not a requirement.

---

## Appendix C — Two corrections from the verify-the-verifier pass

**C1. The lowering is the #1 soundness risk (false unsat).** During evaluation, a mis-nested disjunction in a B-prime encoding produced three spurious `unsat` results — each indistinguishable from a genuine proof. The solver was correct (a minimal `sat` probe confirmed the engine and enum encoding were flawless); the bug was purely in the lowering. Lesson: a verdict is only as sound as the `spec-yaml`→IR→B′ lowering layer. The rule `unknown is not success` must extend to: an `unsat` produced from an untrusted lowering is not a proof until the lowering itself is validated — a lowering bug silently mints false proofs, strictly worse than `unknown`. Implication: the lowering layer needs round-trip or property-preserving checks (or a small trusted kernel); this is the top engineering risk for an NS-Spec implementation, ahead of headless-CI and executable-cell security.

**C2. Label §16.7 preservation vs §16.8 reachability distinctly (SHIP downgrade).** The §16.7 invariant-preservation check runs over all invariant-satisfying states, not just reachable ones. For ORDER-LIFECYCLE it returns `sat` (counterexample `captured=0`): `INV-SHIPPED-PAID` is non-inductive, because SHIP from `paid` does not require `captured>0`. This is a code smell, not a runtime defect — the violating state `paid` with `captured=0` is unreachable (the only transition into `paid` is PAY, which sets `captured=amount>0`). §16.7 and §16.8 return different verdicts by design; a spec can fail §16.7 yet be safe. Rule: the proof report MUST label which check produced a verdict, and never present a §16.7 `sat` as unsafe without a reachability verdict. (Reachability here is by inspection of the transition relation; a solver proof needs a correct unrolling — which is easy to mis-encode, see C1.)